# GAIA Vision — entraînement complet (16 plantes : olivier · palmier dattier · agrumes + 13 cultures PlantVillage)
Notebook pour **Google Colab** (Runtime ▸ Change runtime type ▸ **GPU**). Il fait tout : données → entraînement → calibration → export ONNX → `manifest.json` prêt à copier dans `models/gaia-vision/` de GAIA.

**Plantes** : olivier, palmier dattier, agrumes (jeux dédiés) + pommier, myrtillier, cerisier, maïs, vigne, pêcher, poivron, pomme de terre, framboisier, soja, courge, fraisier, tomate (PlantVillage, téléchargé automatiquement).

**Avant de lancer** :
1. Clé API Roboflow gratuite (olivier) — ou zips téléchargés à la main.
2. Zip du dataset palmier (Mendeley, manuel) — optionnel : sans lui, le palmier est refusé comme « autre ».
3. Le reste est automatique (Hugging Face + torchvision).

⚠️ PlantVillage = feuilles photographiées sur fond uni en laboratoire. Le modèle sera bien meilleur sur ces photos que sur des photos de champ. Teste sur de vraies photos de Tunisie et corrige avec GAIA Kaizen.

## 1. Installation

In [ ]:
!pip -q install roboflow datasets onnx onnxruntime scikit-learn pandas pillow
import torch; print('GPU :', torch.cuda.is_available())

## 2. Scripts du projet (écrits dans `scripts/`)

In [ ]:
import os; os.makedirs('scripts', exist_ok=True); os.makedirs('models_out', exist_ok=True)

In [ ]:
%%writefile scripts/01_prepare.py
"""Build data/index.csv from raw/<tree>/<class>/*.jpg  (any number of trees).
Usage:  python scripts/01_prepare.py --raw raw
 - every sub-folder of raw/ is one plant ("tree" in the manifest): olive, palm, citrus, tomato, potato, ...
 - the folder 'other_crops' is the 'other' class (things GAIA must refuse)
Split is deterministic by file hash -> 80/10/10; files named <base>__<n>.jpg share the split of <base>
(augmented copies of one photo never leak across train/test).
"""
import argparse, collections, csv, hashlib, re
from pathlib import Path

EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
COMMON = {"healthy": "healthy", "buena": "healthy", "sana": "healthy", "health": "healthy", "healthy_sample": "healthy"}
OLIVE = {"aculus": "aculus_olearius", "scab": "peacock_spot", "olive_peacock_spot": "peacock_spot", "knot": "knot_disease"}

def norm(name, tree):
    n = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    n = re.sub(r"^\d+_", "", n)   # Mendeley palm folders are numbered: "9. Healthy sample" -> healthy_sample
    if n in COMMON: return COMMON[n]
    return OLIVE.get(n, n) if tree == "olive" else n   # olive aliases must not touch other crops

def split_of(path: Path):
    h = int(hashlib.md5(path.name.split("__")[0].encode()).hexdigest(), 16) % 100
    return "train" if h < 80 else "val" if h < 90 else "test"

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw", type=Path, default=Path("raw")); ap.add_argument("--out", type=Path, default=Path("data/index.csv"))
    a = ap.parse_args(); rows = []
    for tdir in sorted(p for p in a.raw.iterdir() if p.is_dir()):
        tree = "other" if tdir.name == "other_crops" else tdir.name
        for cdir in sorted(p for p in tdir.iterdir() if p.is_dir()):
            for f in cdir.rglob("*"):
                if f.suffix.lower() in EXT:
                    rows.append([str(f.resolve()), tree, norm(cdir.name, tree), split_of(f)])
    a.out.parent.mkdir(parents=True, exist_ok=True)
    with open(a.out, "w", newline="") as fh:
        w = csv.writer(fh); w.writerow(["path", "tree", "disease", "split"]); w.writerows(rows)
    c = collections.Counter(r[1] for r in rows)
    for k, v in sorted(c.items()): print(f"{k:12s} {v}")
    print("total", len(rows), "->", a.out)

if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/02_train.py
"""Train a MobileNetV3 classifier.
  python scripts/02_train.py --task tree
  python scripts/02_train.py --task disease --tree olive
"""
import argparse, json
from pathlib import Path
import pandas as pd, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from sklearn.metrics import classification_report

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
SIZE = 224

class DS(Dataset):
    def __init__(s, df, classes, col, tf): s.df, s.c2i, s.col, s.tf = df.reset_index(drop=True), {c: i for i, c in enumerate(classes)}, col, tf
    def __len__(s): return len(s.df)
    def __getitem__(s, i):
        r = s.df.iloc[i]
        return s.tf(Image.open(r.path).convert("RGB")), s.c2i[r[s.col]]

def tfs():
    tr = T.Compose([T.RandomResizedCrop(SIZE, scale=(0.35, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                    T.RandomRotation(25), T.ColorJitter(0.35, 0.35, 0.3, 0.05), T.ToTensor(), T.Normalize(MEAN, STD), T.RandomErasing(p=0.25)])
    # same preprocessing as the GAIA page (health.js): centre square crop, then resize
    ev = T.Compose([T.Resize(SIZE), T.CenterCrop(SIZE), T.ToTensor(), T.Normalize(MEAN, STD)])
    return tr, ev

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--task", choices=["tree", "disease"], required=True)
    ap.add_argument("--tree"); ap.add_argument("--index", default="data/index.csv")
    ap.add_argument("--epochs", type=int, default=8); ap.add_argument("--bs", type=int, default=32)
    a = ap.parse_args()
    df = pd.read_csv(a.index)
    if a.task == "tree": col, name = "tree", "tree"
    else: df, col, name = df[df.tree == a.tree], "disease", f"disease_{a.tree}"
    classes = sorted(df[col].unique()); print(name, classes, len(df), "images")
    tr_tf, ev_tf = tfs()
    ds = {s: DS(df[df.split == s], classes, col, tr_tf if s == "train" else ev_tf) for s in ["train", "val", "test"]}
    dl = {s: DataLoader(d, batch_size=a.bs, shuffle=(s == "train"), num_workers=4) for s, d in ds.items()}
    dev = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
    m.classifier[2] = nn.Dropout(0.5)   # anti-overfitting: stronger dropout
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, len(classes)); m.to(dev)
    cnt = df[df.split == "train"][col].value_counts()
    w = torch.tensor([len(df) / (len(classes) * cnt[c]) for c in classes], dtype=torch.float32).to(dev)
    lossf = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
    opt = torch.optim.AdamW(m.parameters(), lr=2e-4, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, a.epochs)

    def evaluate(split):
        m.eval(); P, Y = [], []
        with torch.no_grad():
            for x, y in dl[split]: P += m(x.to(dev)).argmax(1).cpu().tolist(); Y += y.tolist()
        return sum(p == y for p, y in zip(P, Y)) / max(len(Y), 1), P, Y

    best, bad = 0, 0; Path("models_out").mkdir(exist_ok=True)
    for ep in range(a.epochs):
        m.train()
        for x, y in dl["train"]:
            opt.zero_grad(); lossf(m(x.to(dev)), y.to(dev)).backward(); opt.step()
        sched.step(); acc, _, _ = evaluate("val"); print(f"epoch {ep+1} val_acc {acc:.3f}")
        if acc > best:
            best, bad = acc, 0; torch.save({"state": m.state_dict(), "classes": classes}, f"models_out/{name}.pt")
        else:
            bad += 1
            if bad >= 3: print("early stop"); break   # no val gain for 3 epochs
    m.load_state_dict(torch.load(f"models_out/{name}.pt")["state"])
    acc, P, Y = evaluate("test"); print("TEST acc", acc)
    print(classification_report(Y, P, target_names=classes, zero_division=0))

if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/03_calibrate_export.py
"""Temperature-scale on the val split, pick a refusal threshold, export ONNX.
  python scripts/03_calibrate_export.py --name tree
  python scripts/03_calibrate_export.py --name disease_olive
Refusal rule: GAIA answers only if max softmax(logits/T) >= threshold, where the
threshold is the lowest value giving >= --target accuracy on the accepted val images.
"""
import argparse, json
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
import importlib.util, sys
spec = importlib.util.spec_from_file_location("t", "scripts/02_train.py"); t = importlib.util.module_from_spec(spec); spec.loader.exec_module(t)

def logits_of(m, df, classes, col, bs=64):
    dl = DataLoader(t.DS(df, classes, col, t.tfs()[1]), batch_size=bs)
    L, Y = [], []
    with torch.no_grad():
        for x, y in dl: L.append(m(x)); Y.append(y)
    return torch.cat(L), torch.cat(Y)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", required=True); ap.add_argument("--target", type=float, default=0.95)
    ap.add_argument("--index", default="data/index.csv")
    a = ap.parse_args()
    ck = torch.load(f"models_out/{a.name}.pt"); classes = ck["classes"]
    m = models.mobilenet_v3_large(weights=None)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, len(classes)); m.load_state_dict(ck["state"]); m.eval()
    df = pd.read_csv(a.index)
    if a.name == "tree": col = "tree"
    else: df, col = df[df.tree == a.name.split("_", 1)[1]], "disease"
    Lv, Yv = logits_of(m, df[df.split == "val"], classes, col)
    logT = torch.zeros(1, requires_grad=True); opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad(); l = nn.functional.cross_entropy(Lv / logT.exp(), Yv); l.backward(); return l
    opt.step(closure); T_ = float(logT.exp())
    conf = torch.softmax(Lv / T_, 1).max(1).values.numpy(); ok = (Lv.argmax(1) == Yv).numpy()
    thr = 1.0
    for th in np.linspace(0.3, 0.99, 70):
        sel = conf >= th
        if sel.sum() >= 0.2 * len(ok) and ok[sel].mean() >= a.target: thr = float(th); break
    Lt, Yt = logits_of(m, df[df.split == "test"], classes, col)
    ct = torch.softmax(Lt / T_, 1).max(1).values.numpy(); okt = (Lt.argmax(1) == Yt).numpy(); sel = ct >= thr
    rep = {"temperature": T_, "test_accuracy_all": float(okt.mean()),
           "test_coverage": float(sel.mean()), "test_accuracy_when_answering": float(okt[sel].mean()) if sel.any() else None}
    print(rep)
    torch.onnx.export(m, torch.zeros(1, 3, t.SIZE, t.SIZE), f"models_out/{a.name}.onnx", input_names=["input"],
                      output_names=["logits"], dynamic_axes={"input": {0: "n"}, "logits": {0: "n"}}, opset_version=17)
    fname = "stage1-tree.onnx" if a.name == "tree" else f"{a.name.split('_', 1)[1]}.onnx"
    import os; os.replace(f"models_out/{a.name}.onnx", f"models_out/{fname}")
    # keys match what js/health.js reads: file, classes, temperature, threshold
    json.dump({"file": fname, "classes": classes, "temperature": T_, "threshold": thr, "eval": rep},
              open(f"models_out/{a.name}.json", "w"), indent=2)

if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/04_build_manifest.py
"""Assemble the manifest in the schema js/health.js expects, and copy ONNX files next to it.
  python scripts/04_build_manifest.py --dest ../../models/gaia-vision --version 2026.11-v1
Run from ml/gaia-vision. Guidance entries come from advice.json and are included ONLY if validated_by is filled.
"""
import argparse, json, shutil
from pathlib import Path

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dest", type=Path, default=Path("../../models/gaia-vision"))
    ap.add_argument("--version", default="2026.11-v1")
    a = ap.parse_args(); src = Path("models_out"); a.dest.mkdir(parents=True, exist_ok=True)
    pick = lambda j: {k: j[k] for k in ("file", "classes", "temperature", "threshold")}
    man = {"version": a.version,
           "input": {"size": 224, "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
           "stage1": pick(json.load(open(src / "tree.json"))), "stage2": {}, "guidance": {}, "evaluation": {}}
    for f in sorted(src.glob("disease_*.json")):
        j = json.load(open(f)); tree = f.stem.split("_", 1)[1]
        man["stage2"][tree] = pick(j); man["evaluation"][tree] = j["eval"]
    adv = Path("advice.json")
    if adv.exists():
        for k, v in json.load(open(adv)).items():
            if k.startswith("_") or not isinstance(v, dict): continue
            if v.get("validated_by") and v.get("fr"): man["guidance"][k] = {x: v[x] for x in ("fr", "ar", "validated_by", "date") if x in v}
    for f in src.glob("*.onnx"): shutil.copy(f, a.dest / f.name)
    json.dump(man, open(a.dest / "manifest.json", "w"), indent=2, ensure_ascii=False)
    print("written", a.dest / "manifest.json", "| guidance entries:", len(man["guidance"]))

if __name__ == "__main__":
    main()


In [ ]:
%%writefile advice.json
{
  "_note": "Key = <tree>/<disease>. An entry is shipped ONLY when validated_by is filled by an agronomist. Trees: olive, palm, citrus.",
  "olive/peacock_spot": {"fr": "", "ar": "", "validated_by": "", "date": ""},
  "olive/aculus_olearius": {"fr": "", "ar": "", "validated_by": "", "date": ""},
  "olive/healthy": {"fr": "", "ar": "", "validated_by": "", "date": ""}
}


## 3. Données
Structure visée : `raw/<arbre>/<classe>/*.jpg` avec arbres = `olive`, `palm`, `citrus`, `other_crops`.

### 3.0 Remise à zéro des données
**À exécuter à chaque fois** avant les cellules de données : sinon, si tu relances le notebook sans redémarrer la session, les images s’ajoutent une 2ᵉ fois dans `raw/` (c’est arrivé pour le palmier : classes doublées, mêmes photos dans train et test).

In [ ]:
import shutil
for d in ("raw", "dl", "data"): shutil.rmtree(d, ignore_errors=True)
print("raw/, dl/, data/ remis à zéro")

### 3a. Agrumes (automatique) — Citrus Fruits and Leaves Dataset, CC BY 4.0

In [ ]:
from datasets import load_dataset, concatenate_datasets
from pathlib import Path
import random
def save_img(img, path):
    path.parent.mkdir(parents=True, exist_ok=True); img.convert("RGB").save(path, quality=92)

dd = load_dataset("Project-AgML/citrus_fruit_leaf_disease_classification")
ds = concatenate_datasets([dd[k] for k in dd])
lab, part = ds.features["label"].names, ds.features["plant_part"].names
n = 0
for i, ex in enumerate(ds):
    if part[ex["plant_part"]] != "leaf": continue          # feuilles seulement
    save_img(ex["image"], Path("raw/citrus")/lab[ex["label"]]/f"citrus_{i}.jpg"); n += 1
print("agrumes (feuilles) :", n, "| classes :", lab)

### 3b. Les 13 cultures PlantVillage (automatique)
Chaque classe est limitée à 400 images (équilibre + rapidité). `Orange___Haunglongbing` est rangée dans **citrus/greening**. Les images « fond sans feuille » vont dans la classe *autre*.

In [ ]:
import re
CAP = 400
dd = load_dataset("Project-AgML/plant_village_classification")
ds = concatenate_datasets([dd[k] for k in dd]); names = ds.features["label"].names
labels = ds["label"]
slug = lambda x: re.sub(r"[^a-z0-9]+", "_", x.lower()).strip("_")
bucket = {}
for i, l in enumerate(labels): bucket.setdefault(l, []).append(i)
random.seed(0); count = {}
for l, ids in bucket.items():
    nm = names[l]; random.shuffle(ids)
    if "___" in nm:
        crop, dis = nm.split("___", 1)
        crop = slug(re.sub(r"\(.*?\)", "", crop)).split("_")[0]; dis = slug(dis)
        if crop == "orange": crop, dis = "citrus", "greening"
        dest = Path("raw")/crop/dis
    else:                                   # ex. Background_without_leaves
        dest = Path("raw/other_crops")/slug(nm)
    for i in ids[:CAP]:
        save_img(ds[i]["image"], dest/f"pv_{i}.jpg")
    count[str(dest)] = min(CAP, len(ids))
print(len(count), "classes PlantVillage enregistrées")
print(sorted({p.split("/")[1] for p in count}))

### 3b-bis. Classe « autre » — plantes/images hors cultures supportées
Oxford Flowers 102 (fleurs, pas des feuilles) : c’est un choix par défaut faible. **Ajoute tes propres photos** (sol, mains, autres feuilles, fruits, intérieur) dans `raw/other_crops/<nom>/` pour que GAIA refuse mieux.

In [ ]:
try:
    from torchvision.datasets import Flowers102
    fl = Flowers102(root="dl/flowers", split="train", download=True)
    for i in range(len(fl)): save_img(fl[i][0], Path("raw/other_crops/flowers")/f"fl_{i}.jpg")
    print("autres (fleurs) :", len(fl))
except Exception as e:
    print("Flowers102 indisponible :", e)

### 3c. Olivier — Roboflow (CC BY 4.0)
Colle ta clé ci-dessous. Si le téléchargement échoue, télécharge les zips à la main (format *Folder Structure*) et décompresse-les dans `dl/`.

In [ ]:
import shutil, os
ROBOFLOW_KEY = ""   # <-- colle ta clé ici
OLIVE_SOURCES = [("baraah", "olive-s-leaf-diseases-tvce5"), ("project-9k7w7", "olive1-ngwty")]

def merge_class_folders(src, dst, tag):
    """dl/<proj>/{train,valid,test}/<classe>/*.jpg -> raw/.../<classe>/<tag>_<base>__<n>.jpg
    (les copies augmentées d'une même photo gardent le même <base> => même split, pas de fuite)."""
    k = 0
    for sp in Path(src).iterdir():
        if not sp.is_dir(): continue
        for cd in sp.iterdir():
            if not cd.is_dir(): continue
            out = Path(dst)/cd.name; out.mkdir(parents=True, exist_ok=True)
            for f in cd.iterdir():
                if f.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    base = f.name.split(".rf.")[0]; k += 1
                    shutil.copy(f, out/f"{tag}_{base}__{k}{f.suffix.lower()}")
    return k

if ROBOFLOW_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_KEY)
    for ws, proj in OLIVE_SOURCES:
        ok = False
        for v in range(1, 8):
            try:
                rf.workspace(ws).project(proj).version(v).download("folder", location=f"dl/{proj}"); ok = True; break
            except Exception as e: last = e
        print(proj, "OK" if ok else f"échec ({last})")
for ws, proj in OLIVE_SOURCES:
    if Path(f"dl/{proj}").exists(): print(proj, merge_class_folders(f"dl/{proj}", "raw/olive", proj[:6]), "images")
print({d.name: len(list(d.glob('*'))) for d in Path("raw/olive").glob("*")} if Path("raw/olive").exists() else "raw/olive vide")

### 3d. Palmier dattier — Mendeley (manuel)
1. Télécharge le dataset : https://data.mendeley.com/datasets/g684ghfxvg/2 (bouton *Download all*).
2. Dans Colab : panneau Fichiers (à gauche) ▸ glisse le zip, ou exécute la cellule ci-dessous et choisis-le.

In [ ]:
from google.colab import files
import zipfile
up = files.upload()                          # choisis le(s) zip(s) Mendeley palmier
Path("dl/palm").mkdir(parents=True, exist_ok=True)
for name in up:
    if name.endswith(".zip"): zipfile.ZipFile(name).extractall("dl/palm")
# copie chaque dossier qui contient directement des images -> raw/palm/<nom_du_dossier>
EXT = {".jpg", ".jpeg", ".png"}; n = 0
for d in Path("dl/palm").rglob("*"):
    if d.is_dir():
        imgs = [f for f in d.iterdir() if f.suffix.lower() in EXT]
        if imgs:
            out = Path("raw/palm")/d.name; out.mkdir(parents=True, exist_ok=True)
            for f in imgs: n += 1; shutil.copy(f, out/f"palm_{f.stem}__{n}{f.suffix.lower()}")
print({d.name: len(list(d.glob('*'))) for d in Path("raw/palm").glob("*")})

### 3d-bis. Tes propres photos (kaki, figuier, jujubier, ou n’importe quelle autre plante)
Pour une plante sans jeu public (ex. **kaki**), ajoute tes photos. Prépare un zip dont la **racine contient un dossier par plante**, puis un sous-dossier par état :
```
gaia_custom.zip
├── kaki/healthy/*.jpg
├── kaki/circular_leaf_spot/*.jpg   (si tu as des feuilles malades)
└── other_crops/mains_sol_interieur/*.jpg   (images à refuser)
```
Kaggle : uploade-le comme Dataset dont le nom contient `custom`. Colab : glisse `gaia_custom.zip` dans le panneau Fichiers. Une plante avec une seule classe est *identifiée* mais n’a pas de diagnostic. Vise au moins ~150 photos de feuilles par plante, prises sur différents arbres, fonds et lumières.

In [ ]:
import zipfile, shutil
from pathlib import Path
EXT = {".jpg", ".jpeg", ".png"}
roots = []
if Path("/kaggle/input").exists(): roots += [p for p in Path("/kaggle/input").glob("*custom*") if p.is_dir()]
for z in [*Path(".").glob("*custom*.zip"), *Path("/kaggle/input").rglob("*custom*.zip")] if Path("/kaggle/input").exists() else Path(".").glob("*custom*.zip"):
    zipfile.ZipFile(z).extractall("dl/custom"); roots.append(Path("dl/custom")); print("décompressé", z.name)
n = 0
for root in roots:
    for pdir in [p for p in root.iterdir() if p.is_dir()]:
        for cdir in [c for c in pdir.iterdir() if c.is_dir()]:
            out = Path("raw")/pdir.name/cdir.name; out.mkdir(parents=True, exist_ok=True)
            for f in cdir.rglob("*"):
                if f.suffix.lower() in EXT: n += 1; shutil.copy(f, out/f"custom_{f.stem}__{n}{f.suffix.lower()}")
print("photos perso :", n)

### 3e. Vérifier les classes
Regarde les noms de dossiers : si une classe est un doublon ou hors sujet (ex. images de fruits), supprime/renomme le dossier avant de continuer. Garde au moins ~100 images par classe.

In [ ]:
for p in sorted(Path("raw").glob("*")):
    print(p.name, {d.name: len(list(d.glob("*"))) for d in sorted(p.glob("*"))})

### 3f. Supprimer les doublons exacts (toutes sources confondues)
Une même photo présente deux fois (ex. Roboflow + Kaggle, ou relance du notebook) peut tomber dans *train* et dans *test* : le score devient trop beau. Cette cellule garde un seul exemplaire de chaque image.

In [ ]:
import hashlib
seen, removed = {}, 0
for f in sorted(Path("raw").rglob("*")):
    if f.is_file():
        h = hashlib.md5(f.read_bytes()).hexdigest()
        if h in seen: f.unlink(); removed += 1
        else: seen[h] = f
print("doublons exacts supprimés :", removed, "| images uniques :", len(seen))
# anti-overfitting : max 1200 images par classe (les gros jeux olivier/palmier contiennent beaucoup de copies quasi identiques)
import random; random.seed(0); CAPC = 1200; cut = 0
for cd in [d for t_ in Path("raw").glob("*") for d in t_.glob("*") if d.is_dir()]:
    fs = sorted(cd.glob("*"))
    if len(fs) > CAPC:
        for f in random.sample(fs, len(fs) - CAPC): f.unlink(); cut += 1
print("images retirées pour équilibrer :", cut)
for p in sorted(Path("raw").glob("*")):
    print(p.name, {d.name: len(list(d.glob("*"))) for d in sorted(p.glob("*"))})

## 4. Index + découpage train/val/test

In [ ]:
!python scripts/01_prepare.py --raw raw
import pandas as pd
df = pd.read_csv("data/index.csv"); df = df[df.tree != "other"]
# une plante n'a un modèle « maladie » que si elle a au moins 2 classes (ex. myrtillier = 1 seule classe => identifié, sans diagnostic)
TREES = sorted(t for t, g in df.groupby("tree") if g.disease.nunique() >= 2)
print("modèles maladie à entraîner :", TREES)
print("plantes sans modèle maladie :", sorted(set(df.tree) - set(TREES)))

## 5. Entraînement — étape 1 : arbre

In [ ]:
!python scripts/02_train.py --task tree --epochs 8

## 6. Entraînement — étape 2 : une maladie par arbre

In [ ]:
for tree in TREES:
    print("=" * 20, tree)
    get_ipython().system(f"python scripts/02_train.py --task disease --tree {tree} --epochs 8")

## 7. Calibration, seuil de refus, export ONNX
Le seuil de refus = valeur minimale de confiance pour laquelle ≥ 95 % des réponses sont justes sur la validation. Si `threshold` = 1.0, le modèle n’atteint pas la cible : il refusera tout → il faut plus/meilleures données.

In [ ]:
!python scripts/03_calibrate_export.py --name tree
for tree in TREES:
    if Path(f"models_out/disease_{tree}.pt").exists():
        print("=" * 20, tree)
        get_ipython().system(f"python scripts/03_calibrate_export.py --name disease_{tree}")

## 8. Manifest + téléchargement
`advice.json` : seules les fiches avec `validated_by` rempli par un agronome sont publiées (sinon GAIA affiche « aucune fiche validée »).

In [ ]:
!python scripts/04_build_manifest.py --dest out/models/gaia-vision --version 2026.11-v1
!cat out/models/gaia-vision/manifest.json
!cd out && zip -qr ../gaia-vision-models.zip models && ls -lh ../gaia-vision-models.zip
from google.colab import files; files.download("gaia-vision-models.zip")

## 9. Brancher dans GAIA
Dézippe `gaia-vision-models.zip` à la racine du dépôt GAIA → `models/gaia-vision/manifest.json` + les `.onnx`.
Lance un serveur local (la page lit le manifest par `fetch`, donc pas en `file://`), recharge `/health` : les étapes 2 à 6 doivent passer de « Modèle non connecté » à actives.

**Ensuite** : mesure sur un jeu de test tunisien (tes propres photos) et remplis `evaluation` dans le manifest avant toute démonstration publique.